# Customer Segmentation with Actionable Business Insights

**Goal:** Segment customers using RFM (Recency, Frequency, Monetary) analysis and K-Means clustering,
then translate each segment into concrete, actionable business recommendations (retention offers,
upsell targets, win-back campaigns, VIP treatment, etc.)

---
### 📌 Dataset
This notebook is built for the **Online Retail dataset** (UCI Machine Learning Repository), a
transactional dataset from a UK-based online retailer.

**Paste your dataset link / path here:**
```
DATASET_URL = "PASTE_YOUR_DATASET_URL_HERE"
```
Suggested source (free, ready to use):
👉 https://archive.ics.uci.edu/dataset/352/online+retail
(or the Kaggle mirror: https://www.kaggle.com/datasets/carrie1/ecommerce-data)

The notebook works with any transactional dataset that has these columns
(rename yours to match, or edit the `COLUMN_MAP` in Step 2):
- `CustomerID` – unique customer identifier
- `InvoiceNo` – order / transaction identifier
- `InvoiceDate` – date of transaction
- `Quantity` – units purchased
- `UnitPrice` – price per unit

---
### 🧭 Notebook Structure
1. Setup & Imports
2. Load Data (paste your dataset URL/path)
3. Data Cleaning
4. Exploratory Data Analysis (EDA)
5. RFM Feature Engineering
6. Scaling
7. Finding Optimal Number of Clusters (Elbow + Silhouette)
8. K-Means Clustering
9. Cluster Visualization (PCA)
10. Cluster Profiling
11. **Actionable Business Insights per Segment**
12. Export Results


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.max_columns", 50)


## 2. Load Data

👉 **Paste your dataset URL or local path in `DATASET_URL` below.**

Works with:
- A direct CSV/Excel download link
- A path to an uploaded file (e.g. `/content/data.csv` in Colab)
- The UCI Online Retail `.xlsx` file


In [ ]:
# ============================================
# PASTE YOUR DATASET URL / PATH HERE
# ============================================
DATASET_URL = "PASTE_YOUR_DATASET_URL_HERE"   # e.g. "https://.../online_retail.csv" or "/content/online_retail.xlsx"

# If you're in Google Colab and want to upload a file manually, uncomment:
# from google.colab import files
# uploaded = files.upload()
# DATASET_URL = list(uploaded.keys())[0]

# Load CSV or Excel automatically based on file extension
if DATASET_URL.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(DATASET_URL)
else:
    df = pd.read_csv(DATASET_URL, encoding="ISO-8859-1")

print("Shape:", df.shape)
df.head()


### Column mapping
If your dataset uses different column names, map them here so the rest of the notebook works unchanged.

In [ ]:
COLUMN_MAP = {
    "CustomerID": "CustomerID",
    "InvoiceNo": "InvoiceNo",
    "InvoiceDate": "InvoiceDate",
    "Quantity": "Quantity",
    "UnitPrice": "UnitPrice",
}
df = df.rename(columns={v: k for k, v in COLUMN_MAP.items() if v in df.columns})
df.info()


## 3. Data Cleaning

In [ ]:
# Drop rows without a CustomerID (can't segment anonymous transactions)
df = df.dropna(subset=["CustomerID"])

# Remove cancelled orders (InvoiceNo starting with 'C') and non-positive quantities/prices
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

# Parse date, create total price per line item
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

# Drop exact duplicate rows
df = df.drop_duplicates()

print("Cleaned shape:", df.shape)
df.head()


## 4. Exploratory Data Analysis

In [ ]:
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
print("Unique customers:", df["CustomerID"].nunique())
print("Unique invoices:", df["InvoiceNo"].nunique())

monthly_revenue = df.set_index("InvoiceDate").resample("ME")["TotalPrice"].sum()
monthly_revenue.plot(kind="line", marker="o", title="Monthly Revenue")
plt.ylabel("Revenue")
plt.show()


In [ ]:
top_customers = df.groupby("CustomerID")["TotalPrice"].sum().sort_values(ascending=False).head(10)
top_customers.plot(kind="bar", title="Top 10 Customers by Spend")
plt.ylabel("Total Spend")
plt.show()


## 5. RFM Feature Engineering

- **Recency (R):** days since the customer's last purchase (lower = more recently active)
- **Frequency (F):** number of distinct purchases (higher = more loyal)
- **Monetary (M):** total amount spent (higher = more valuable)


In [ ]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum")
).reset_index()

print(rfm.shape)
rfm.describe()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(rfm["Recency"], bins=30, ax=axes[0]).set_title("Recency")
sns.histplot(rfm["Frequency"], bins=30, ax=axes[1]).set_title("Frequency")
sns.histplot(rfm["Monetary"], bins=30, ax=axes[2]).set_title("Monetary")
plt.tight_layout()
plt.show()


## 6. Scaling\nRFM values are skewed and on very different scales, so we log-transform and standardize them before clustering.

In [ ]:
rfm_log = rfm[["Recency", "Frequency", "Monetary"]].apply(lambda x: np.log1p(x))

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)
rfm_scaled = pd.DataFrame(rfm_scaled, columns=["Recency", "Frequency", "Monetary"])
rfm_scaled.head()


## 7. Finding the Optimal Number of Clusters

In [ ]:
inertia = []
silhouette = []
K_range = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertia.append(km.inertia_)
    silhouette.append(silhouette_score(rfm_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_range), inertia, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")

axes[1].plot(list(K_range), silhouette, marker="o", color="green")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Score")
plt.tight_layout()
plt.show()


## 8. K-Means Clustering\nPick `k` based on the elbow/silhouette plots above (4 is a common, business-friendly choice for RFM).

In [ ]:
OPTIMAL_K = 4  # <-- adjust based on the plots above

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

rfm["Cluster"].value_counts().sort_index()


## 9. Cluster Visualization (PCA)

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_result = pca.fit_transform(rfm_scaled)
rfm["PC1"], rfm["PC2"] = pca_result[:, 0], pca_result[:, 1]

plt.figure(figsize=(8, 6))
sns.scatterplot(data=rfm, x="PC1", y="PC2", hue="Cluster", palette="tab10", s=40)
plt.title("Customer Segments (PCA projection)")
plt.show()


## 10. Cluster Profiling

In [ ]:
cluster_profile = rfm.groupby("Cluster").agg(
    Customers=("CustomerID", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Total_Revenue=("Monetary", "sum")
).round(1).sort_values("Avg_Monetary", ascending=False)

cluster_profile


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(data=cluster_profile.reset_index(), x="Cluster", y="Avg_Recency", ax=axes[0])
sns.barplot(data=cluster_profile.reset_index(), x="Cluster", y="Avg_Frequency", ax=axes[1])
sns.barplot(data=cluster_profile.reset_index(), x="Cluster", y="Avg_Monetary", ax=axes[2])
axes[0].set_title("Avg Recency by Cluster")
axes[1].set_title("Avg Frequency by Cluster")
axes[2].set_title("Avg Monetary by Cluster")
plt.tight_layout()
plt.show()


## 11. Actionable Business Insights per Segment

Use the cluster profile table above to **label each cluster** and attach a concrete action.
A common RFM-based labeling scheme is shown below — adjust the thresholds/labels to match
what you actually see in `cluster_profile`.

| Segment (typical pattern) | RFM Signature | Business Action |
|---|---|---|
| **Champions / VIPs** | Low Recency, High Frequency, High Monetary | Reward with loyalty perks, early access to new products, personal account manager. Protect this segment — they drive disproportionate revenue. |
| **Loyal / Steady Customers** | Low-Med Recency, High Frequency, Med Monetary | Upsell/cross-sell complementary products, invite to a loyalty program, ask for reviews/referrals. |
| **At-Risk / Needs Attention** | High Recency, Med-High past Frequency, Med Monetary | Launch a win-back email campaign with a personalized discount before they churn completely. |
| **Lost / Dormant Customers** | Very High Recency, Low Frequency, Low Monetary | Low-cost reactivation campaign (email only); if no response after 2 attempts, deprioritize spend on this group. |
| **New / Low-Value Customers** | Low Recency, Low Frequency, Low Monetary | Nurture with onboarding content and a second-purchase incentive to move them toward "Loyal." |

**How to use this section:**
1. Run the cell below — it prints the actual stats for each cluster in *your* data.
2. Match each cluster to the closest pattern in the table above (or redefine your own).
3. Fill in `SEGMENT_LABELS` with names that make sense for your business, then re-run.


In [ ]:
# Inspect actual cluster stats to decide labels
for c in cluster_profile.index:
    row = cluster_profile.loc[c]
    print(f"Cluster {c}: {int(row.Customers)} customers | "
          f"Avg Recency={row.Avg_Recency} days | "
          f"Avg Frequency={row.Avg_Frequency} orders | "
          f"Avg Monetary=${row.Avg_Monetary} | "
          f"Total Revenue=${row.Total_Revenue}")


In [ ]:
# ============================================
# EDIT THIS: map each cluster number to a business-friendly segment name
# ============================================
SEGMENT_LABELS = {
    0: "Champions / VIPs",
    1: "Loyal Customers",
    2: "At-Risk Customers",
    3: "Lost / Dormant Customers",
}

rfm["Segment"] = rfm["Cluster"].map(SEGMENT_LABELS)

revenue_share = (rfm.groupby("Segment")["Monetary"].sum() / rfm["Monetary"].sum() * 100).round(1)
customer_share = (rfm.groupby("Segment")["CustomerID"].count() / len(rfm) * 100).round(1)

insight_summary = pd.DataFrame({
    "Customer % of Base": customer_share,
    "Revenue % of Total": revenue_share
}).sort_values("Revenue % of Total", ascending=False)

insight_summary


### 💡 Example takeaways to report to stakeholders
- *"X% of customers (Champions) generate Y% of total revenue — prioritize retention budget here."*
- *"Z% of customers are At-Risk; a targeted win-back campaign could recover an estimated $__ in revenue."*
- *"Lost customers now make up N% of the base — consider suppressing marketing spend on this group and reallocating it to New Customers."*

Fill in the blanks using the numbers from `insight_summary` and `cluster_profile` above.


## 12. Export Results

In [ ]:
rfm.to_csv("customer_segments.csv", index=False)
print("Saved customer_segments.csv with", len(rfm), "customers and their assigned segment.")
rfm.head()
